### Evaluate models on economic metrics

In [ ]:
import sys

sys.path.append("../../src/")


import os
import pickle

import numpy as np
from lightning import seed_everything

from model_evaluation.trading_strategies import (
    trading_fixed_hours,
    trading_optimal_bids,
    trading_quantile_strategy,
    trading_unlimited_bids,
)
from model_training.data_modules.utils import EPFDataModule

In [ ]:
seed_everything(0)

#### Load model results including predictions and quantiles

In [ ]:
models_dict = []

In [ ]:
# file_path = "results/metric_evaluation/NaiveHS"
# file_path = (
#     "results/metric_evaluation/DDNN_Ens"
# )
# file_path = (
#     "results/metric_evaluation/MCD"
# )
# file_path = "results/metric_evaluation/EvDNN"
# file_path = "results/metric_evaluation/LEAR_GARCH"
# file_path = "results/metric_evaluation/LEAR_QRA"
# file_path = "results/metric_evaluation/LEAR_CP"
# file_path = "results/metric_evaluation/DDNN_CP"
# file_path = "results/metric_evaluation/Ens_CP"
# file_path = "results/metric_evaluation/MCD_CP"
# file_path = "results/metric_evaluation/EvDNN_CP"
# file_path = "results/metric_evaluation/XGBoost_GARCH"
# file_path = "results/metric_evaluation/XGBoost_QRA"
file_path = "results/metric_evaluation/XGBoost_CP"

In [ ]:
with open(
    file_path + ".pkl",
    "rb",
) as f:
    models_dict.extend(pickle.load(f))

In [ ]:
file_path_trading = (
    file_path.replace("metric_evaluation", "economical_evaluation") + "_trading"
)
if os.path.exists(file_path_trading + ".pkl"):
    models_dict_trading = pickle.load(open(file_path_trading + ".pkl", "rb"))
    # combine dicts
    for i, model in enumerate(models_dict_trading):
        model.update(models_dict[i])
    models_dict = models_dict_trading

#### Load dataset

In [ ]:
standardization_case = "mean_std"

In [ ]:
# Load the data
val_date = "2022-12-01"
test_date = "2023-12-01"
end_date = "2024-11-30"
data_file_path = "../../data/processed/smard_data_201810010000_202501010000.npz"
data_module = EPFDataModule(
    data_file_path=data_file_path,
    val_date=val_date,
    test_date=test_date,
    end_date=end_date,
    batch_size=32,
    standardization_case=standardization_case,
)

train_input, train_labels = data_module.train_dataset[:]
val_input, val_labels = data_module.val_dataset[:]
test_input, test_labels = data_module.test_dataset[:]

data_input, data_labels = test_input, test_labels

data_labels = data_labels * data_module.scale_target + data_module.offset_target
data_labels = data_labels.detach().numpy()

#### Perform different strategies

In [ ]:
t = trading_optimal_bids(data_labels)
print(np.sum(t["profit_order"]))
print(np.mean(t["profit_order"]))

In [ ]:
t = trading_fixed_hours(
    data_labels,
    (val_labels * data_module.scale_target + data_module.offset_target)
    .detach()
    .numpy(),
)
print(np.sum(t["profit_order"]))
print(np.mean(t["profit_order"]))

In [ ]:
os.path.exists(file_path_trading + ".pkl")

In [ ]:
# quantile trading strategy
if not os.path.exists(file_path_trading + ".pkl"):
    for model in models_dict:
        model["quantile_trading"] = [
            list(
                reversed(
                    [
                        trading_quantile_strategy(
                            prediction_interval=model["quantile"][m, [i, -i - 1], :, :],
                            prediction_price_day_ahead=model["prediction"][m, :, :],
                            price_day_ahead=data_labels,
                        )
                        for i in range(model["quantile"].shape[1] // 2)
                    ]
                )
            )
            for m in range(model["quantile"].shape[0])
        ]
        print("Doneeee")

In [ ]:
# calculate the profit and per-transaction profit
for model in models_dict:
    model["quantile_trading_profit"] = []
    model["quantile_trading_profit_per_day"] = []
    model["quantile_trading_ptp"] = []
    model["quantile_trading_profitble_trades"] = []

    for i in range(len(model["quantile_trading"])):
        conf_ptp_list = []
        conf_profit_list = []
        conf_profit_per_day_list = []
        conf_profitable_trades_list = []

        for j in range(len(model["quantile_trading"][i])):
            days_list = []
            days_list_per_day = []
            days_profitalbe_trade_counter = 0
            for k in range(len(model["quantile_trading"][i][j])):
                days_list.extend(model["quantile_trading"][i][j][k]["profit_order"])
                days_list_per_day.append(
                    np.sum(model["quantile_trading"][i][j][k]["profit_order"])
                )
                days_profitalbe_trade_counter += (
                    1
                    if (model["quantile_trading"][i][j][k]["make_limit_orders"])
                    else 0
                )
            conf_profit_list.append(np.sum(days_list))
            conf_profit_per_day_list.append(
                np.stack(
                    days_list_per_day + [0]
                    if len(days_list_per_day) != 366
                    else days_list_per_day
                )
            )
            conf_ptp_list.append(np.mean(days_list))
            conf_profitable_trades_list.append(
                days_profitalbe_trade_counter / len(model["quantile_trading"][i][j])
            )
        model["quantile_trading_profit"].append(conf_profit_list)
        model["quantile_trading_profit_per_day"].append(
            np.stack(conf_profit_per_day_list)
        )
        model["quantile_trading_ptp"].append(conf_ptp_list)
        model["quantile_trading_profitble_trades"].append(conf_profitable_trades_list)
    model["quantile_trading_profit"] = np.array(model["quantile_trading_profit"])
    model["quantile_trading_ptp"] = np.array(model["quantile_trading_ptp"])
    model["quantile_trading_profitble_trades"] = np.array(
        model["quantile_trading_profitble_trades"]
    )

    model["quantile_trading_profit_mean"] = np.mean(
        model["quantile_trading_profit"], axis=0
    )
    model["quantile_trading_profit_std"] = np.std(
        model["quantile_trading_profit"], axis=0
    )

    model["quantile_trading_profit_per_day_mean"] = np.mean(
        model["quantile_trading_profit_per_day"], axis=0
    )
    model["quantile_trading_profit_per_day_std"] = np.std(
        model["quantile_trading_profit_per_day"], axis=0
    )

    model["quantile_trading_ptp_mean"] = np.mean(model["quantile_trading_ptp"], axis=0)
    model["quantile_trading_ptp_std"] = np.std(model["quantile_trading_ptp"], axis=0)

    model["quantile_trading_profitble_trades_mean"] = np.mean(
        model["quantile_trading_profitble_trades"], axis=0
    )
    model["quantile_trading_profitble_trades_std"] = np.std(
        model["quantile_trading_profitble_trades"], axis=0
    )

In [ ]:
# unlimited bids trading strategy
for model in models_dict:
    model["unlimited_bids_trading"] = [
        trading_unlimited_bids(
            prediction_price_day_ahead=model["prediction"][m, :, :],
            price_day_ahead=data_labels,
        )
        for m in range(model["prediction"].shape[0])
    ]

In [ ]:
# calculate the profit and per-transaction profit
for model in models_dict:
    model["unlimited_bids_profit"] = []
    model["unlimited_bids_profit_per_day"] = []
    model["unlimited_bids_ptp"] = []
    model["unlimited_bids_profitble_trades"] = []

    for i in range(len(model["unlimited_bids_trading"])):
        days_list = []
        days_list_per_day = []
        days_profitalbe_trade_counter = 0
        for k in range(len(model["unlimited_bids_trading"][i])):
            days_list.extend(model["unlimited_bids_trading"][i][k]["profit_order"])
            days_list_per_day.append(
                np.sum(model["unlimited_bids_trading"][i][k]["profit_order"])
            )
            days_profitalbe_trade_counter += (
                1
                if (model["unlimited_bids_trading"][i][k]["prediction_profit"] > 0)
                else 0
            )
        model["unlimited_bids_profit"].append(np.sum(days_list))
        model["unlimited_bids_profit_per_day"].append(
            np.stack(
                days_list_per_day + [0]
                if len(days_list_per_day) != 366
                else days_list_per_day
            )
        )
        model["unlimited_bids_ptp"].append(np.mean(days_list))
        model["unlimited_bids_profitble_trades"].append(
            days_profitalbe_trade_counter / len(model["unlimited_bids_trading"][i])
        )
    model["unlimited_bids_profit"] = np.array(model["unlimited_bids_profit"])
    model["unlimited_bids_profit_per_day"] = np.array(
        model["unlimited_bids_profit_per_day"]
    )
    model["unlimited_bids_ptp"] = np.array(model["unlimited_bids_ptp"])
    model["unlimited_bids_profitble_trades"] = np.array(
        model["unlimited_bids_profitble_trades"]
    )

    model["unlimited_bids_profit_mean"] = np.mean(
        model["unlimited_bids_profit"], axis=0
    )
    model["unlimited_bids_profit_std"] = np.std(model["unlimited_bids_profit"], axis=0)

    model["unlimited_bids_profit_per_day_mean"] = np.mean(
        model["unlimited_bids_profit_per_day"], axis=0
    )
    model["unlimited_bids_profit_per_day_std"] = np.std(
        model["unlimited_bids_profit_per_day"], axis=0
    )

    model["unlimited_bids_ptp_mean"] = np.mean(model["unlimited_bids_ptp"], axis=0)
    model["unlimited_bids_ptp_std"] = np.std(model["unlimited_bids_ptp"], axis=0)

    model["unlimited_bids_profitble_trades_mean"] = np.mean(
        model["unlimited_bids_profitble_trades"], axis=0
    )
    model["unlimited_bids_profitble_trades_std"] = np.std(
        model["unlimited_bids_profitble_trades"], axis=0
    )

In [ ]:
# save trading results in extra dict
models_dict_trading = []
for model in models_dict:
    model_trading = {}
    model_trading["model_name"] = model["model_name"]

    model_trading["quantile_trading"] = model["quantile_trading"]

    model_trading["quantile_trading_profit"] = model["quantile_trading_profit"]
    model_trading["quantile_trading_profit_mean"] = model[
        "quantile_trading_profit_mean"
    ]
    model_trading["quantile_trading_profit_std"] = model["quantile_trading_profit_std"]

    model_trading["quantile_trading_profit_per_day"] = model[
        "quantile_trading_profit_per_day"
    ]
    model_trading["quantile_trading_profit_per_day_mean"] = model[
        "quantile_trading_profit_per_day_mean"
    ]
    model_trading["quantile_trading_profit_per_day_std"] = model[
        "quantile_trading_profit_per_day_std"
    ]

    model_trading["quantile_trading_ptp"] = model["quantile_trading_ptp"]
    model_trading["quantile_trading_ptp_mean"] = model["quantile_trading_ptp_mean"]
    model_trading["quantile_trading_ptp_std"] = model["quantile_trading_ptp_std"]
    model_trading["quantile_trading_profitble_trades"] = model[
        "quantile_trading_profitble_trades"
    ]
    model_trading["quantile_trading_profitble_trades_mean"] = model[
        "quantile_trading_profitble_trades_mean"
    ]
    model_trading["quantile_trading_profitble_trades_std"] = model[
        "quantile_trading_profitble_trades_std"
    ]

    model_trading["unlimited_bids_trading"] = model["unlimited_bids_trading"]

    model_trading["unlimited_bids_profit"] = model["unlimited_bids_profit"]
    model_trading["unlimited_bids_profit_mean"] = model["unlimited_bids_profit_mean"]
    model_trading["unlimited_bids_profit_std"] = model["unlimited_bids_profit_std"]

    model_trading["unlimited_bids_profit_per_day"] = model[
        "unlimited_bids_profit_per_day"
    ]
    model_trading["unlimited_bids_profit_per_day_mean"] = model[
        "unlimited_bids_profit_per_day_mean"
    ]
    model_trading["unlimited_bids_profit_per_day_std"] = model[
        "unlimited_bids_profit_per_day_std"
    ]

    model_trading["unlimited_bids_ptp"] = model["unlimited_bids_ptp"]
    model_trading["unlimited_bids_ptp_mean"] = model["unlimited_bids_ptp_mean"]
    model_trading["unlimited_bids_ptp_std"] = model["unlimited_bids_ptp_std"]

    model_trading["unlimited_bids_profitble_trades"] = model[
        "unlimited_bids_profitble_trades"
    ]
    model_trading["unlimited_bids_profitble_trades_mean"] = model[
        "unlimited_bids_profitble_trades_mean"
    ]
    model_trading["unlimited_bids_profitble_trades_std"] = model[
        "unlimited_bids_profitble_trades_std"
    ]

    models_dict_trading.append(model_trading)

In [ ]:
# # save
# file_path = file_path.replace("metric_evaluation", "economical_evaluation")
# if not os.path.exists(file_path + "_trading" + ".pkl"):
#     with open(
#         file_path + "_trading" + ".pkl",
#         "wb",
#     ) as f:
#         pickle.dump(models_dict_trading, f)
# else:
#     print("File already exists, not overwriting")

In [ ]:
# save
file_path = file_path.replace("metric_evaluation", "economical_evaluation")
with open(
    file_path + "_trading" + ".pkl",
    "wb",
) as f:
    pickle.dump(models_dict_trading, f)

In [ ]:
models_dict_trading[0].keys()